# Targeted customer retention proposal — Company A

Prepared by **Hossam_Elsheikh**. This notebook reproduces the EDA, LightGBM validation, ranking metrics, and scenario evidence used in the presentation. It uses only the supplied Company A data.

## Setup
Extract the course archive so `Client.csv` and `Record.csv` are inside a `telecom/` folder beside this notebook, or change `DATA_DIR` below. Install the packages in `requirements.txt`.

In [ ]:
from pathlib import Path
import pandas as pd

possible = [Path("telecom"), Path("Final Assignment/telecom"), Path("data/telecom")]
DATA_DIR = next((p for p in possible if (p / "Client.csv").exists()), possible[0])
print("Using:", DATA_DIR.resolve())


## Load and join the source tables
The source documents describe one row per customer in each table. The join is validated as one-to-one; the customer ID is dropped before modeling.

In [ ]:
client = pd.read_csv(DATA_DIR / "Client.csv")
record = pd.read_csv(DATA_DIR / "Record.csv")
df = record.merge(client, on="Customer_ID", how="inner", validate="one_to_one")
print("Client:", client.shape, "Record:", record.shape, "Merged:", df.shape)
print("Observed churn rate:", round(df["churn"].mean(), 4))


## Exploratory evidence
The following grouped churn rates describe associations in the observed sample; they do not show causal effects.

In [ ]:
age_bins = pd.cut(df["eqpdays"], [-1, 90, 180, 365, 730, 99999],
                  labels=["<=90", "91-180", "181-365", "366-730", ">730"], include_lowest=True)
print(df.groupby(age_bins, observed=False)["churn"].agg(["size", "mean"]).rename(columns={"mean":"churn_rate"}))
print("\nMean usage change by outcome:")
print(df.groupby("churn")["change_mou"].mean())
print("\nMean monthly revenue by outcome:")
print(df.groupby("churn")["rev_Mean"].mean())


## Train and evaluate the risk model
LightGBM handles the mixed numeric and categorical fields. A stratified holdout provides the reported ROC-AUC and average precision. The training script then refits a five-fold ensemble and saves feature importance and validation predictions.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "telecom_analysis.py", "--data-dir", str(DATA_DIR), "--output", "output"], check=True)


## Review results
The scenario table is intentionally illustrative. It assumes a 12-month horizon and an offer cost of 10 dataset currency units. The supplied materials do not identify currency, margin, actual campaign cost, or treatment uplift. Validate those inputs in a randomized pilot before budgeting.

In [ ]:
import json
metrics = json.loads(Path("output/telecom_metrics.json").read_text())
metrics


In [ ]:
importance = pd.read_csv("output/telecom_feature_importance.csv")
importance.head(15)


## References and submission note
See `Hossam_Elsheikh_references.txt` for the full source list. The course requires the share URL for any generative-AI chat in Omnicampus References; add the URL for this conversation before submitting.